# P-values
The coin-tossing p-value with a slider (move the observed number of heads), and p-values for the two z-tests.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy import stats

## 1. Coin tossed 100 times: p-value for k heads
Under H0 the number of heads is Binomial(100, 0.5); H1 says heads come up more often.

In [ ]:
coin = stats.binom(n=100, p=0.5)
for k in [53, 60, 65, 80]:
    # sf(k - 1) = P(X > k - 1) = P(X >= k)
    print(f"{k} heads: P(X = {k}) = {coin.pmf(k):.3g}, p-value P(X >= {k}) = {coin.sf(k - 1):.3g}")
print(f"P(40 <= X <= 60) = {coin.cdf(60) - coin.cdf(39):.3f}")

## 2. Interactive: move the observed number of heads
Red bars are the results as or more extreme than the observed one; their total is the p-value.

In [ ]:
ks = np.arange(25, 86)
fig = go.Figure()
steps = []
observed = list(range(50, 82, 2))
for i, k in enumerate(observed):
    colours = np.where(ks >= k, "#E45756", "#4C78A8")
    fig.add_bar(x=ks, y=coin.pmf(ks), marker_color=colours, visible=(i == 0), showlegend=False)
    vis = [j == i for j in range(len(observed))]
    steps.append(dict(method="update", label=str(k),
                      args=[{"visible": vis}, {"title": f"{k} heads: p-value = {coin.sf(k - 1):.4g}"}]))
fig.update_layout(template="simple_white", title=f"50 heads: p-value = {coin.sf(49):.4g}",
                  sliders=[dict(steps=steps, currentvalue={"prefix": "observed heads: "})],
                  xaxis_title="number of heads in 100 tosses", yaxis_title="probability under H0")
fig.show()

## 3. Check the meaning by simulation
Repeat the 100-toss experiment many times with a fair coin: how often do we get 53 or more heads?

In [ ]:
rng = np.random.default_rng(0)
heads = rng.binomial(100, 0.5, size=100_000)
print("share of experiments with >= 53 heads:", np.mean(heads >= 53))
print("share with exactly 53 heads:          ", np.mean(heads == 53))

## 4. P-values for the z-tests

In [ ]:
# training program: right-tailed
z1 = (53 - 50) / (5 / np.sqrt(30))
p1 = stats.norm.sf(z1)
print(f"training: z = {z1:.2f}, p = {p1:.4f}, reject H0: {p1 <= 0.05}")
# chips packets: two-tailed
z2 = (49 - 50) / (4 / np.sqrt(40))
p2 = 2 * stats.norm.cdf(-abs(z2))
print(f"chips:    z = {z2:.2f}, p = {p2:.3f}, reject H0: {p2 <= 0.05}")
# p-values of other z values: the strength of evidence
for z in [1.7, 2, 15]:
    print(f"z = {z}: right-tailed p = {stats.norm.sf(z):.3g}")